# 멀티에이전트: 전문가 팀 조율하기

Claude Managed Agents와 멀티에이전트 코디네이터 패턴으로, 중견 운영 조직에 워크플로 자동화 플랫폼을 파는 가상의 회사 Northstar의 영업 제안서 작성을 자동화해 보겠습니다.

지금 이 회사의 영업 담당자들은 잠재 고객마다 맞춤 제안서를 만듭니다. 해당 고객이 속한 세그먼트의 기업들이 보통 무엇을 중시하는지 조사하고, 수백 건의 라이브러리에서 관련 사례 연구 두 건을 뽑고, 내부 규칙 시트로 가격을 산정하고, 이를 두 쪽짜리 문서로 조립합니다. 각 단계가 서로 다른 자료와 서로 다른 종류의 판단을 요구합니다.

코디네이터 에이전트가 전문가 셋을 돌려 이 일을 하게 하겠습니다. 리서처는 웹 검색으로 잠재 고객 세그먼트의 일반적인 양상을 찾습니다. 사서는 사례 연구 라이브러리를 읽고 가장 잘 맞는 두 건을 고릅니다. 가격 모델러는 규칙 파일과 좌석 수만 봅니다. 코디네이터가 이들의 순서를 정하고 제안서를 씁니다.

## 1. 클라이언트 설정

먼저 SDK를 설치하고 Anthropic 클라이언트를 설정합니다. 멀티에이전트 설정과 이벤트 타입은 Managed Agents 베타의 일부입니다.

In [1]:
%%capture
%pip install -q anthropic python-dotenv

In [2]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-opus-4-6")
client = anthropic.Anthropic()

## 2. 세 전문가 서브에이전트 정의하기

다음으로 팀원 셋을 만듭니다. 각자 고유한 시스템 프롬프트, 고유한 출력 형태, 그리고 자기 일에 필요한 도구만 갖습니다. 리서처는 웹 검색을, 사례 연구 선별자는 로컬 라이브러리 읽기만, 가격 모델러는 `pricing_rules.md`와 좌석 수만 봅니다. 역할마다 도구를 좁혀 두면 가격 담당이 웹에서 경쟁사 수치를 끌어오는 일을 막고, 사례 연구 라이브러리 전체가 코디네이터의 컨텍스트에 들어오지 않게 됩니다.

In [3]:
def make_agent(name, description, system, tools):
    a = client.beta.agents.create(
        name=name,
        description=description,
        model=MODEL,
        system=system,
        tools=tools,
        betas=BETAS,
    )
    print(f"{name}: {a.id}")
    return a.id


prospect_researcher = make_agent(
    "prospect_researcher",
    "Researches what companies in a given industry segment and size tier typically prioritize.",
    """Given a prospect's industry and size, use web search to find:
- What companies in that segment typically list as strategic priorities
- Recent trends or pressures in that industry
- Common operational pain points at that scale
Return via send_to_parent: {"priorities": [...], "recent_moves": [...], "pain_points": [...], "sources": [...]}""",
    [
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
)

case_study_picker = make_agent(
    "case_study_picker",
    "Selects the two most relevant case studies from the library for a given prospect profile.",
    """The case study library is in /mnt/user-data/case_studies/. Each file is one customer story.
You will be given a prospect's industry, size, and top priorities. Read the library, score each study on relevance, and pick the two best matches.
Return via send_to_parent: {"picks": [{"file": ..., "customer": ..., "why_relevant": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

pricing_modeler = make_agent(
    "pricing_modeler",
    "Builds two or three pricing options for a prospect based on seat count and expected usage.",
    """Pricing rules are in /mnt/user-data/pricing_rules.md. Given a prospect's estimated seat count and usage tier, build:
- a conservative option (annual commit, lower per-seat)
- a flexible option (monthly, higher per-seat)
- if seat count > 500, an enterprise option with a platform fee
Show the first-year total for each. Return via send_to_parent: {"options": [{"name": ..., "structure": ..., "year_one_total": ...}, ...]}""",
    [{"type": "agent_toolset_20260401"}],
)

prospect_researcher: agent_011Cahy5YQwAN99heAeSpPob
case_study_picker: agent_011Cahy5ZKWFqEAgPNZZxPvM


pricing_modeler: agent_011Cahy5a5eTdwuzZabFgu6K


## 3. 팀에 다룰 자료 주기

사서에게는 고를 라이브러리가 필요합니다. 헬스케어, 제조, 물류, 소매, 핀테크, 공공 부문에 걸친 짧은 사례 연구 일곱 건을 제공해, 사서가 실제로 우리 잠재 고객에 맞는 두 건을 고르는 모습을 볼 수 있게 하겠습니다.

In [4]:
CASE_STUDIES = [
    {
        "slug": "stclair_health",
        "title": "St. Clair Health",
        "industry": "regional hospital network",
        "employees": 6200,
        "summary": """Challenge: credentialing and prior-auth workflows spread across 11 systems.
Result with Northstar: consolidated to 3 automated workflows; prior-auth turnaround down 58%; $1.9M annual labor savings.""",
    },
    {
        "slug": "blueridge_health_plan",
        "title": "BlueRidge Health Plan",
        "industry": "regional payer",
        "employees": 2800,
        "summary": """Challenge: claims-adjudication exceptions queued in email; 19% required manual rework.
Result with Northstar: exception routing automated end-to-end; rework rate down to 6%; 11-day faster average claim resolution.""",
    },
    {
        "slug": "calder_mfg",
        "title": "Calder Manufacturing",
        "industry": "industrial",
        "employees": 3100,
        "summary": """Challenge: purchase-order approvals averaging 9 days.
Result with Northstar: PO cycle time cut to 2.1 days; 14% reduction in maverick spend.""",
    },
    {
        "slug": "northwind",
        "title": "Northwind Logistics",
        "industry": "3PL",
        "employees": 4400,
        "summary": """Challenge: carrier-onboarding paperwork took 3 weeks per carrier.
Result with Northstar: onboarding down to 4 days; 22% more carriers activated in Q1.""",
    },
    {
        "slug": "harborview_retail",
        "title": "Harborview Retail Group",
        "industry": "specialty retail",
        "employees": 5600,
        "summary": """Challenge: store-level inventory exceptions handled by regional managers over Slack and spreadsheets.
Result with Northstar: exception triage automated across 140 stores; stockout incidents down 31%.""",
    },
    {
        "slug": "aperture_fintech",
        "title": "Aperture Payments",
        "industry": "fintech",
        "employees": 1900,
        "summary": """Challenge: KYC and merchant-onboarding reviews averaging 6 business days.
Result with Northstar: review SLA cut to 36 hours; onboarding throughput up 2.4x with the same team.""",
    },
    {
        "slug": "summit_county",
        "title": "Summit County Government",
        "industry": "public sector",
        "employees": 3700,
        "summary": """Challenge: building-permit applications routed through five departments by paper packet.
Result with Northstar: single digital intake with parallel department review; median permit time 41 to 17 days.""",
    },
]

### 제품·가격 자료

코디네이터가 "어떻게 도움이 되는가" 섹션을 쓸 때 읽는 제품 한 장 요약과, 모델러가 옵션을 구성할 때 사용하는 가격 규칙 파일도 함께 제공합니다.

In [5]:
PRODUCT = """# Northstar Platform — One-Pager
Northstar is a workflow automation platform for mid-market operations teams.
Core capabilities: visual process builder, 200+ SaaS connectors, role-based approvals, SOC 2 Type II.
Typical results: 40-60% reduction in manual ticket handling, 3-week time-to-first-workflow."""

PRICING = """# Pricing Rules (internal)
- Per-seat list: $65/mo (monthly) or $52/mo (annual commit).
- Usage tiers: light = 1.0x, standard = 1.15x, heavy = 1.30x multiplier on per-seat.
- Enterprise (>500 seats): add $48,000/yr platform fee, per-seat drops to $44/mo annual.
- All options include onboarding; enterprise includes a named CSM."""

### 코디네이터 연결하고 세션 시작하기

이제 환경을 만들고 파일 아홉 개를 업로드한 뒤, 전문가 셋으로 이뤄진 `multiagent` 명단을 갖춘 코디네이터를 만듭니다. 각 항목은 저마다 모델, 프롬프트, 도구 세트를 갖춘 완전한 에이전트이므로, 역할별로 모델 등급을 섞어 쓸 수도 있습니다.

In [6]:
env = client.beta.environments.create(
    name="proposal-meridian",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

resources = []


def mount(path, content):
    f = client.beta.files.upload(
        file=(os.path.basename(path), content.encode(), "text/plain")
    )
    resources.append({"type": "file", "file_id": f.id, "mount_path": path})


for cs in CASE_STUDIES:
    body = f"# {cs['title']} ({cs['industry']}, {cs['employees']:,} employees)\n{cs['summary']}"
    mount(f"/mnt/user-data/case_studies/{cs['slug']}.md", body)
mount("/mnt/user-data/product_one_pager.md", PRODUCT)
mount("/mnt/user-data/pricing_rules.md", PRICING)

coordinator = client.beta.agents.create(
    name="Proposal Writer",
    model=MODEL,
    system="""You assemble tailored sales proposals.
Given a prospect name and basic profile:
1. Send the prospect's industry and size to prospect_researcher.
2. Send the prospect's industry, size, and (once the researcher reports back) their priorities to case_study_picker.
3. Send the seat count and usage tier to pricing_modeler.
4. Read /mnt/user-data/product_one_pager.md, then write /mnt/session/outputs/proposal.md with sections:
   Executive summary (tied to their priorities), How we help (from the one-pager),
   Proof (the two case studies), Investment (the pricing options), Next steps.
Keep it to two pages.""",
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [prospect_researcher, case_study_picker, pricing_modeler],
    },
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": coordinator.id, "version": coordinator.version},
    environment_id=env.id,
    resources=resources,
    title="Proposal: Meridian Health",
    betas=BETAS,
)
print(f"Session {session.id} ready with {len(resources)} files mounted")

Session sesn_011Cahy5wq6Jyk32n2NwgR88 ready with 9 files mounted


## 4. 제안서 작업 시작하기

잠재 고객 프로필을 보내고 코디네이터가 일하는 모습을 지켜보겠습니다. 리서처와 가격 모델러를 병렬로 시작한 뒤, 리서처의 결과가 돌아오면 사례 연구 선별자를 실행합니다. 선별자가 관련성을 매기려면 그 우선순위 정보가 필요하기 때문입니다.

In [7]:
PROSPECT = {
    "name": "Meridian Health",
    "industry": "regional healthcare system",
    "employees": 8500,
    "estimated_seats": 600,
    "usage_tier": "heavy",
}

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": f"Build a proposal for {PROSPECT['name']}, a {PROSPECT['industry']} with "
                    f"~{PROSPECT['employees']} employees. Estimate {PROSPECT['estimated_seats']} seats "
                    f"at {PROSPECT['usage_tier']} usage. Write to /mnt/session/outputs/proposal.md.",
                }
            ],
        }
    ],
)

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "session.thread_created":
            print(f"[spawn] {ev.agent_name}")
        elif ev.type == "agent.thread_message_received":
            print(f"[report] {ev.from_agent_name} returned")
        elif ev.type == "session.status_idle":
            print("[done]")
            break

[spawn] prospect_researcher


[spawn] pricing_modeler


[report] prospect_researcher returned


[spawn] case_study_picker


[report] pricing_modeler returned


[report] case_study_picker returned


[done]


### 팀원들이 보내온 것

조립된 제안서를 보기 전에, 세 개의 원시 `send_to_parent` 페이로드를 출력해 보겠습니다. 각 서브에이전트가 자신의 도구만 가진 자신만의 컨텍스트에서 실행되었기 때문에, 세 보고서는 서로 꽤 다르게 보입니다.

In [8]:
def text_of(content):
    return "".join(b.text for b in content if b.type == "text")


for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if ev.type == "agent.thread_message_received":
        body = text_of(ev.content)
        print(f"━━━ send_to_parent from {ev.from_agent_name} ({len(body)} chars) ━━━")
        print(body[:1200] + (f"\n…[{len(body) - 1200} more chars]" if len(body) > 1200 else ""))
        print()

━━━ send_to_parent from prospect_researcher (9580 chars) ━━━
Here is the structured research for tailoring a sales proposal to Meridian Health (regional healthcare system, ~8,500 employees):

---

## 1. TOP 3–5 STRATEGIC PRIORITIES FOR REGIONAL HEALTHCARE SYSTEMS IN 2025–2026

**Priority 1: Financial Resilience & Cost Containment**
- Regional health systems are under intense financial pressure from reimbursement shortfalls, labor cost inflation, and policy changes (e.g., Medicaid cuts, 340B reforms under the One Big Beautiful Bill Act). Per Plante Moran: "Medicaid changes could create tens of millions in revenue shortfalls for some systems." Cash flow acceleration, revenue cycle optimization, and cost control without compromising care quality are top-of-mind.
- EY notes: "Reimbursement, labor shortage and cost inflation headwinds are compounded by the need to invest in leading clinical programs." Regional health systems specifically are "signaling the strategic importance of increasing

## 5. 제안서 읽기

마지막으로 조립된 제안서를 가져옵니다. 코디네이터가 `write` 도구로 `proposal.md`에 썼으므로, 로그에서 해당 이벤트를 찾아 어떤 섹션을 만들었는지 살펴보겠습니다. 문서 전체를 읽고 싶다면 `proposal` 자체를 출력하세요.

In [9]:
proposal = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input["file_path"].endswith("proposal.md")
    ):
        proposal = ev.input["content"]
        break

# Show the section structure rather than the full proposal.
for line in proposal.splitlines():
    if line.startswith("#"):
        print(line)

# Northstar Platform — Proposal for Meridian Health
## Executive Summary
## How We Help
## Proof — Results from Healthcare Organizations Like Yours
### St. Clair Health — Regional Hospital Network, 6,200 Employees
### BlueRidge Health Plan — Regional Health Plan, 2,800 Employees
## Investment — Three Options for 600 Seats, Heavy Usage
## Next Steps


## 서브에이전트를 하나가 아니라 셋으로 나눈 이유

세 도구를 모두 가진 에이전트 하나로도 이 제안서를 쓸 수 있는데 왜 나눴을까요? 역할마다 도구를 좁혀 두면 가격 모델러는 규칙 파일만 갖고 있으므로 웹에서 경쟁사 정가를 끌어올 수 없습니다. 사례 연구 선별자는 여기서 파일 일곱 개를 읽지만 실제 환경에서는 수백 건을 읽게 되는데, 그 분량이 코디네이터가 아니라 서브에이전트의 컨텍스트에 머무릅니다. 그리고 코디네이터는 전문가의 일을 직접 하지 않으면서 순서와 인계 시점을 결정할 수 있습니다.

멀티에이전트 조율에 대해 더 알아보려면 [Managed Agents 문서](https://platform.claude.com/docs/en/managed-agents/multi-agent)를 참고하세요.